# Mean-level season contrasts and the innings-total tail

Threads 1 to 3 of the 2023 vs 2024 contrast: phase scoring and pace share (Thread 1), the bookended-acceleration shape (Thread 2), and the innings-total tail (Thread 3). Everything runs off `scripts/mean_level_and_tail/`; this notebook only orchestrates, displays, and writes outputs with their provenance sidecars.

What is computed here:

- **Theme table** (Thread 1 plus the harness-wide themes). Match cluster bootstrap on one shared set of resamples. Differences are primary; ratios run on the log scale and are back-transformed. Primary interval is bootstrap-t (closed-form cluster SE) for rates, rate differences and contrasts; BCa with a season-stratified jackknife elsewhere. The table carries both, plus the raw-scale BCa for ratio themes so the two constructions sit side by side, and a log-ratio bias flag.
- **Shape contrasts** (Thread 2). Curvature contrast (+1, -2, +1 across phases) in boundary share, six share and run rate, and middle-against-league in run rate, on shared draws, with equivalence verdicts against the SESOI values in `config.toml`.
- **Innings tail** (Thread 3). First-innings population with a full allowance (chases are censored at the target); failed chases added as the sensitivity population. 220+ share as a 2x2 risk difference (Newcombe interval, Fisher exact), shift function on a fixed quantile grid, exceedances at 180/200/220, and a GPD tail fit above 180 when the exceedance gate passes.

Seeds, resample counts, interval levels and SESOI values come from `config.toml`. Every table written below gets a `.meta.json` sidecar recording them.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("scripts").resolve()))

from common.config import load_settings
from common import results_io
from mean_level_and_tail import season_contrast_bootstrap as scb
from mean_level_and_tail.phase_shape_contrasts import run_shape_contrasts
from mean_level_and_tail.innings_tail import run_tail_analysis, plot_shift_function
from mean_level_and_tail import forest_plot
from mean_level_and_tail.regression_check import run_regression_check

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

settings = load_settings()
ANALYSIS = "mean_level_and_tail"
SEED = settings.seed(ANALYSIS)
N_ITER = settings.n_resamples
PRODUCER = "mean_level_and_tail.ipynb"
print(f"root={settings.root}  seed={SEED}  resamples={N_ITER}  level={settings.interval_level}")

## Data

The cleaned two-season frame (`data/dpl_prelims_clean.parquet`). If it is absent the schema-faithful synthetic fixture stands in so the pipeline can be exercised; every number then is mechanics only and nothing from it is a finding. `USING_FIXTURE` guards the cells that only make sense on real data.

In [ ]:
DATA_PATH = settings.data_path("cleaned_dataset")
USING_FIXTURE = not DATA_PATH.exists()
if USING_FIXTURE:
    from common.synthetic_ballframe import make_synthetic_ballframe
    df = make_synthetic_ballframe(seed=7)
    print("REAL DATA ABSENT: running on the synthetic fixture (mechanics only, no findings)")
else:
    df = scb.load_cleaned_dataset(str(DATA_PATH))
print(f"rows={len(df):,}  matches={df['p_match'].nunique()}  seasons={sorted(df['year'].unique())}")

## Schema confirmations

Cheap checks that precede any rate. Over numbers must run 1 to 20 (a 0-indexed export shifts every phase boundary); `wide` is a run count and `noball` a flag; byes and leg byes are populated so a dot must key off `batruns`; `bowl_kind` carries a small mixture/unknown mass; `target` is NaN in the first innings; `control_clean` must be present.

In [ ]:
print("over_num range:", df["over_num"].agg(["min", "max"]).tolist())
display(df[["wide", "noball", "byes", "legbyes"]].describe().loc[["mean", "max"]])
display(df.groupby("year")["bowl_kind"].value_counts(normalize=True).round(4).unstack(fill_value=0))
print("outcome values:", sorted(df["outcome"].astype(str).unique()))
print("target NaN share, inns 1 / inns 2:",
      round(df.loc[df["inns"] == 1, "target"].isna().mean(), 3),
      round(df.loc[df["inns"] == 2, "target"].isna().mean(), 3))
print("control_clean present:", "control_clean" in df.columns)
print("max_balls below 120 anywhere (innings):",
      int((df.groupby(["p_match", "inns"])["max_balls"].min() < 120).sum()))

## Theme table

One shared set of match resamples for every theme. Columns: `estimate`; `ci_low`/`ci_high` with `interval_method` naming the primary construction; `bca_*` and `boot_t_*` always; `raw_scale_bca_*` for ratio themes; `se_linearized` (closed-form cluster SE) and `se_boot`; `log_ratio_bias` and its flag. The two all-innings tail themes at the bottom are censored and superseded by the tail section; they stay in the table for continuity with the regression check.

In [ ]:
theme_table, theme_results = scb.run_all(df, n_iter=N_ITER, seed=SEED,
                                         alpha=1.0 - settings.interval_level, return_results=True)
results_io.write_table(theme_table, settings, ANALYSIS, "theme_contrasts", SEED, PRODUCER,
                       extra={"n_resamples": N_ITER, "fixture": USING_FIXTURE})
theme_table

## Precision-stability guard

Balls per team in each phase, by season. A between-team spread is sensitive to a change in estimation precision, so this is read before any dispersion result.

In [ ]:
for name in ("powerplay", "middle", "death"):
    for year in (settings.earlier_season, settings.later_season):
        counts = scb.balls_per_team_in_phase(df[df["year"] == year], settings.phase(name))
        print(f"{name:10s} {year}: min={counts.min()}  median={int(counts.median())}  max={counts.max()}")

## Resample distribution of one theme

The replicates behind an interval, for inspecting skew, multimodality, and where the mass sits relative to the null. Ratio themes are shown on the log scale, the scale they are estimated on.

In [ ]:
INSPECT = "powerplay_six_rate_ratio"
r = next(t for t in theme_results if t.name == INSPECT)
reps = r.replicates[np.isfinite(r.replicates)]
null = np.log(r.null_value) if r.scale == "log" else r.null_value
plt.figure(figsize=(6.5, 3.2))
plt.hist(reps, bins=60, color="#2F5C8A", alpha=0.85)
plt.axvline(null, color="k", lw=1)
plt.axvline(np.median(reps), color="C1")
plt.title(f"{INSPECT}: bootstrap replicates ({r.scale} scale)")
plt.show()

## Shape contrasts (Thread 2) with equivalence verdicts

Four composite contrasts on shared draws. The verdict table reports, per channel, the estimate, the 90% (TOST) and 95% intervals, the SE, the pre-stated SESOI, the minimum detectable effect at 80% power, and the verdict: `rejects_null`, `equivalent_to_zero`, or `inconclusive`. Shares are in percentage points. `hypothesis_status` is `retired` only if both dense channels (boundary, six) are equivalent to zero.

In [ ]:
shape_table, shape_verdicts = run_shape_contrasts(df, settings, SEED, n_iter=N_ITER)
results_io.write_table(shape_table, settings, ANALYSIS, "shape_contrasts", SEED, PRODUCER,
                       extra={"n_resamples": N_ITER, "fixture": USING_FIXTURE})
results_io.write_table(shape_verdicts, settings, ANALYSIS, "shape_contrast_verdicts", SEED, PRODUCER,
                       extra={"n_resamples": N_ITER, "fixture": USING_FIXTURE,
                              "sesoi": settings.section("sesoi"),
                              "hypothesis_status": shape_verdicts.attrs["hypothesis_status"]})
print("bookended-acceleration hypothesis:", shape_verdicts.attrs["hypothesis_status"])
display(shape_table[["theme", "estimate", "ci_low", "ci_high", "interval_method", "se_linearized", "resampling_stable"]])
shape_verdicts.round(4)

## Innings-total tail (Thread 3)

Population table first: innings per season by census class (DLS reduced, successful chase, failed chase, first innings) and the all-out-before-death count among first innings. Then, for the primary and sensitivity populations: the 2x2 contrasts at 180/200/220 (the 220 row is the headline), the log-scale bootstrap share ratio as a secondary check, the shift function on the quantile grid, the GPD fit, and the previous upper-quantile shifts as an appendix.

In [ ]:
tail = run_tail_analysis(df, settings, SEED, n_iter=N_ITER)
tail_extra = {"n_resamples": N_ITER, "fixture": USING_FIXTURE, "tail": settings.section("tail")}
results_io.write_table(tail["population_counts"], settings, ANALYSIS, "tail_population_counts", SEED, PRODUCER, tail_extra)
results_io.write_table(tail["innings_table"], settings, ANALYSIS, "tail_innings_table", SEED, PRODUCER, tail_extra)
results_io.write_table(tail["threshold_contrasts"], settings, ANALYSIS, "tail_threshold_contrasts", SEED, PRODUCER, tail_extra)
results_io.write_table(tail["share_ratio_bootstrap"], settings, ANALYSIS, "tail_share_ratio_bootstrap", SEED, PRODUCER, tail_extra)
results_io.write_table(tail["shift_function"], settings, ANALYSIS, "tail_shift_function", SEED, PRODUCER, tail_extra)
results_io.write_table(tail["sparse_upper_quantile_shifts"], settings, ANALYSIS, "tail_sparse_upper_quantile_shifts", SEED, PRODUCER, tail_extra)
results_io.write_json(tail["gpd"], settings, ANALYSIS, "tail_gpd", SEED, PRODUCER, tail_extra)
display(tail["population_counts"])
tail["threshold_contrasts"].round(4)

In [ ]:
display(tail["share_ratio_bootstrap"][["population", "theme", "estimate", "ci_low", "ci_high",
                                       "interval_method", "log_ratio_bias", "log_ratio_bias_flag"]])
tail["shift_function"][["population", "quantile", "estimate", "ci_low", "ci_high", "interval_method"]]

In [ ]:
import json
print(json.dumps(tail["gpd"], indent=1, default=float))
tail["sparse_upper_quantile_shifts"][["population", "theme", "estimate", "ci_low", "ci_high"]]

In [ ]:
fig = plot_shift_function(tail["shift_function"], settings.interval_level)
results_io.write_figure(fig, settings, ANALYSIS, "tail_shift_function", SEED, PRODUCER)
plt.show()

## Forest plot of resampling-stable effects

The bootstrap themes that clear their null, panelled by unit, with the first-innings 220+ risk difference appended as a `points` row. The stability flag filters the rows; nothing is hand-picked.

In [ ]:
headline = tail["threshold_contrasts"].query(
    "headline and population == 'first_innings_full_allowance'").iloc[0].to_dict()
combined = pd.concat([theme_table, pd.DataFrame([forest_plot.risk_difference_row(headline)])],
                     ignore_index=True)
fig, _ = forest_plot.draw_forest(forest_plot.rows_from_results(combined),
                                 "Season-over-season effects that survive the bootstrap",
                                 N_ITER, settings.interval_level)
results_io.write_figure(fig, settings, ANALYSIS, "resampling_stable_findings", SEED, PRODUCER)
plt.show()

## Regression check against the previous run

Real data only. Reproduces the previous notebook's stored table with the previous seeds and the pooled jackknife: every point estimate must match exactly and every stored BCa interval must match exactly. The `stratified_*` columns show how far the interval moves under the season-stratified jackknife on the same draws. A mismatch in `estimate_matches` or `interval_matches` means an estimand changed.

In [ ]:
if USING_FIXTURE:
    print("skipped: the reference numbers are from the real data")
else:
    regression = run_regression_check(df, settings, n_iter=10000)
    results_io.write_table(regression, settings, ANALYSIS, "regression_check", SEED, PRODUCER,
                           extra={"reference": "reference_previous_run.csv", "stored_run_seeds": [36, 20240601]})
    print("all points match:", bool(regression["estimate_matches"].all()),
          "| all stored intervals match:", bool(regression["interval_matches"].all()))
    display(regression)